In [ ]:
#| default_exp machine_learning.semantic_search

In [ ]:
#| export
import os
import pathlib
import time  # Add this at the very top of your file

import weaviate
from weaviate.classes.config import Configure, DataType, Property
from weaviate.util import generate_uuid5

import os
import re
from typing import List, Union, Optional, Type, Iterable, Callable


In [ ]:
#| export

def latex_comment_stripping_processor(path: Union[str, os.PathLike]) -> str:
    r"""
    Opens a file and removes LaTeX comments while ignoring escaped percents (\%).
    Targets lines starting with % (not preceded by \) until the end of the line.
    """
    try:
        with open(os.fspath(path), "r", encoding="utf-8") as f:
            text = f.read()
        
        # Regex Breakdown:
        # (?<!\\) -> Lookbehind: Ensure the previous character is NOT a backslash
        # %.      -> Match the % and everything following it on that line
        pattern = r"(?<!\\)%.*"
        
        # Remove comments and strip trailing whitespace from affected lines
        cleaned_text = re.sub(pattern, "", text)
        return cleaned_text
        
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return ""

In [ ]:
#| export
import os
import re
import fnmatch
import weaviate
import hashlib
from typing import List, Union, Optional, Iterable, Callable
from weaviate.classes.config import Configure, Property, DataType, VectorDistances
from weaviate.classes.query import Filter # Added missing import
from weaviate.util import generate_uuid5
from tqdm import tqdm

PathType = Union[str, os.PathLike]
FileProcessor = Callable[[PathType], str]

class MathBrainClient:
    def __init__(
        self, 
        host: str = "localhost", 
        port: int = 8080,
        chunk_size: int = 800,
        overlap: int = 150,
        batch_size: int = 1
    ) -> None:
        self.client = weaviate.connect_to_local(host=host, port=port)
        self.CHUNK_SIZE = chunk_size
        self.OVERLAP = overlap
        self.BATCH_SIZE = batch_size

    def setup_collection(self, collection_name: str, force_recycle: bool = False) -> None:
        exists = self.client.collections.exists(collection_name)
        if force_recycle and exists:
            print(f"Force Recycle: Deleting existing collection {collection_name}...")
            self.client.collections.delete(collection_name)
            exists = False

        if not exists:
            self.client.collections.create(
                name=collection_name,
                vector_config=Configure.Vectors.text2vec_ollama(
                    name="default",
                    api_endpoint="http://ollama:11434",
                    model="nomic-embed-text",
                    vector_index_config=Configure.VectorIndex.hnsw(
                        distance_metric=VectorDistances.COSINE
                    ),
                ),
                properties=[
                    Property(name="content", data_type=DataType.TEXT),
                    Property(name="fileName", data_type=DataType.TEXT),
                    Property(name="filePath", data_type=DataType.TEXT),
                    Property(name="contentHash", data_type=DataType.TEXT), # Added property
                ]
            )
            print(f"Collection `{collection_name}` initialized.")

    def _split_text(self, text: str) -> List[str]:
        text = re.sub(r'\n{3,}', '\n\n', text)
        paragraphs = text.split('\n\n')
        chunks: List[str] = []
        current_chunk = ""

        for para in paragraphs:
            para = para.strip()
            if not para: continue

            if len(para) > self.CHUNK_SIZE:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                    current_chunk = ""
                step = self.CHUNK_SIZE - self.OVERLAP
                for i in range(0, len(para), step):
                    chunks.append(para[i : i + self.CHUNK_SIZE])
                continue

            if len(current_chunk) + len(para) <= self.CHUNK_SIZE:
                current_chunk += (para + "\n\n")
            else:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                overlap_text = current_chunk[-self.OVERLAP:] if len(current_chunk) > self.OVERLAP else ""
                current_chunk = overlap_text + para + "\n\n"

        if current_chunk:
            chunks.append(current_chunk.strip())
        return [c[:self.CHUNK_SIZE].strip() for c in chunks if len(c) > 10]

    def _get_file_hash(self, text: str) -> str:
        return hashlib.md5(text.encode('utf-8')).hexdigest()

    def ingest_files(
        self, 
        input_source: Union[PathType, Iterable[PathType]], 
        collection_name: str = "MathDocument", 
        force_recycle: bool = False, 
        processor: Optional[FileProcessor] = None,
        exclude_patterns: Optional[List[str]] = None
    ) -> None:
        self.setup_collection(collection_name, force_recycle)
        collection = self.client.collections.get(collection_name)
        
        # Setup Processor
        def default_proc(p):
            with open(os.fspath(p), "r", encoding="utf-8") as f: return f.read()
        active_processor = processor or default_proc
        ignores = exclude_patterns or []

        # 1. Gather Files
        all_paths = []
        if isinstance(input_source, (str, os.PathLike)) and os.path.isdir(input_source):
            for root, _, files in os.walk(input_source):
                for f in files:
                    full_p = os.path.join(root, f)
                    if f.lower().endswith((".tex", ".md", ".txt")):
                        if not any(fnmatch.fnmatch(f, pat) or fnmatch.fnmatch(full_p, pat) for pat in ignores):
                            all_paths.append(full_p)
        else:
            all_paths = [os.fspath(p) for p in input_source]

        print(f"Syncing {len(all_paths)} files...")

        pbar = tqdm(all_paths, desc="MathBrain Sync")
        with collection.batch.fixed_size(batch_size=self.BATCH_SIZE) as batch:
            for path in pbar:
                file_name = os.path.basename(path)
                pbar.set_postfix({"file": file_name[:20]})
                
                try:
                    text = active_processor(path)
                    if not text.strip(): continue
                    current_hash = self._get_file_hash(text)

                    # 2. SMART SKIP: Check if file + hash already exists
                    existing = collection.query.fetch_objects(
                        filters=(
                            Filter.by_property("filePath").equal(str(path)) & 
                            Filter.by_property("contentHash").equal(current_hash)
                        ),
                        limit=1,
                        return_properties=[]
                    )
                    
                    if len(existing.objects) > 0 and not force_recycle:
                        continue # File is unchanged, skip it!

                    # 3. CLEANUP: Delete old chunks for this file
                    collection.data.delete_many(
                        where=Filter.by_property("filePath").equal(str(path))
                    )
                    
                    # 4. INDEX: Add new chunks
                    chunks = self._split_text(text)
                    for i, chunk in enumerate(chunks):
                        batch.add_object(
                            properties={
                                "content": chunk,
                                "fileName": file_name,
                                "filePath": str(path),
                                "contentHash": current_hash 
                            },
                            uuid=generate_uuid5(f"{path}_{i}")
                        )
                except Exception as e:
                    print(f"\n[Error] {file_name}: {e}")

        final_count = collection.aggregate.over_all(total_count=True).total_count
        print(f"\nSync Complete. Brain contains {final_count} objects.")

    def close(self): self.client.close()
    def __enter__(self): return self
    def __exit__(self, *args): self.close()

    def delete_collection(self, collection_name: str):
        """Permanent deletion of a collection and all its vectors."""
        if self.client.collections.exists(collection_name):
            self.client.collections.delete(collection_name)
            print(f"Collection '{collection_name}' has been deleted.")
        else:
            print(f"Deletion skipped: '{collection_name}' does not exist.")

In [ ]:
# import random
# if __name__ == "__main__":
#     from pathlib import Path
    
#     # Example using pathlib.Path
#     DOC_DIR = Path(r"C:\Users\hyunj\Documents\Obsidian\Chores\math\_writing")

#     # samples = [
#     #     Path(r"C:\Users\hyunj\Documents\Obsidian\Chores\math\_writing\_definitions\definition_scheme.tex"),
#     #     Path(r"C:\Users\hyunj\Documents\Obsidian\Chores\math\_writing\_definitions\definition_categories_of_presheaves_and_sheaves_on_a_topological_space_valued_in_a_category.tex"),
#     #     Path(r"C:\Users\hyunj\Documents\Obsidian\Chores\math\_writing\_definitions\definition_sheaf_on_a_site.tex"),
#     #     ]

#     # 1. Gather all potential paths first
#     all_eligible_files = []
#     for root, _, files in os.walk(DOC_DIR):
#         for f in files:
#             if f.lower().endswith((".tex", ".md", ".txt")) and f != "main.tex":
#                 all_eligible_files.append(Path(root) / f)

#     # 2. Calculate 1% (minimum of 1 file so it doesn't fail on small dirs)
#     sample_size = max(1, int(len(all_eligible_files) * 0.01))
    
#     # 3. Randomly sample the list
#     random.seed(42)
#     sampled_paths = random.sample(all_eligible_files, sample_size)
    
#     print(f"Total files found: {len(all_eligible_files)}")
#     print(f"Sampling 1% -> {len(sampled_paths)} files for this test run.")


    

#     # You can now tune these parameters here!
#     with MathBrainClient(
#         chunk_size=2000, 
#         overlap=400, 
#         batch_size=10
#     ) as brain:
#         brain.ingest_files(
#             input_source=sampled_paths,
#             # DOC_DIR, 
#             collection_name="math_writing",
#             force_recycle=True, 
#             exclude_patterns=["main.tex"],
#             processor=latex_comment_stripping_processor,
#         )

In [ ]:
# with MathBrainClient() as brain:
#     brain.delete_collection("MathDocument")

In [ ]:
#| export
# import weaviate
# import weaviate.classes.query as wvc
# import weaviate
# from weaviate.classes.query import MetadataQuery

# class MathBrainSearcher:
#     def __init__(
#             self,
#             collection_name: str,
#             host: str = "localhost",
#             port: int = 8080
#             ) -> None:
#         self.client = weaviate.connect_to_local(host=host, port=port)
#         self.collection_name = collection_name
#         self.collection = self.client.collections.get(self.collection_name)
        
#         # Check if we are ready immediately
#         count = self.collection.aggregate.over_all(total_count=True).total_count
#         if count == 0:
#             print("Wait... Brain reports 0 objects. Checking again in 2 seconds...")
#             import time
#             time.sleep(2)

#     def search(self, query_text, limit=3):
#         """Perform a semantic/vector search."""
#         print(f"\nSearching for: '{query_text}'...")
        
#         response = self.collection.query.near_text(
#             query=query_text,
#             limit=limit,
#             return_metadata=wvc.MetadataQuery(distance=True)
#         )

#         if not response.objects:
#             print("No matches found. Is the brain empty?")
#             return

#         for i, obj in enumerate(response.objects):
#             print(f"\n--- Result #{i+1} (Distance: {obj.metadata.distance:.4f}) ---")
#             print(f"File: {obj.properties['fileName']}")
#             print(f"Path: {obj.properties['filePath']}")
#             print("-" * 30)
#             # Print first 500 chars of the content
#             content = obj.properties['content']
#             preview = (content[:500] + '...') if len(content) > 500 else content
#             print(preview)

#     def close(self):
#         self.client.close()

In [ ]:
#| export
import weaviate
import weaviate.classes.query as wvc
import time

class MathBrainSearcher:
    def __init__(self, collection_name: str, host: str = "localhost", port: int = 8080):
        self.client = weaviate.connect_to_local(host=host, port=port)
        self.collection = self.client.collections.get(collection_name)

    def search(self, query: str, alpha: float = 0.5, limit: int = 3, top_k: int = 20, rerank: bool = False):
        """Unified search with score and vector distance tracking."""
        start_time = time.time()
        fetch_count = max(top_k, limit) if rerank else limit

        response = self.collection.query.hybrid(
            query=query,
            alpha=alpha,
            limit=fetch_count,
            rerank=wvc.Rerank(prop="content", query=query) if rerank else None,
            # We now request both Score (Blended) and Distance (Vector Only)
            return_metadata=wvc.MetadataQuery(score=True, distance=True)
        )

        results = response.objects[:limit]
        self._display_summary(len(results), time.time() - start_time)
        self._display_results(results)

    def _display_summary(self, count: int, duration: float):
        print(f"\nFound {count} matches in {duration:.3f} seconds.")
        print("=" * 60)

    def _display_results(self, objects):
        if not objects:
            print("No matches found.")
            return
        for i, obj in enumerate(objects):
            self._print_single_object(i + 1, obj)

    def _print_single_object(self, rank: int, obj):
        props = obj.properties
        # Score: Higher is better | Distance: Lower is better
        score = obj.metadata.score or 0.0
        dist = f"{obj.metadata.distance:.4f}" if obj.metadata.distance is not None else "N/A (Keyword match)"
        
        print(f"Result #{rank}")
        print(f" > Hybrid Score: {score:.4f} (Higher is better)")
        print(f" > Vector Dist:  {dist} (Lower is better)")
        print(f"File: {props.get('fileName')}")
        print("-" * 30)
        
        content = props.get('content', "")
        preview = (content[:500] + '...') if len(content) > 500 else content
        print(f"{preview}\n")

    def close(self):
        self.client.close()

    def __enter__(self): return self
    def __exit__(self, *args): self.close()

# import weaviate
# import weaviate.classes.query as wvc
# import time

# class MathBrainSearcher:
#     def __init__(self, collection_name: str, host: str = "localhost", port: int = 8080):
#         self.client = weaviate.connect_to_local(host=host, port=port)
#         self.collection = self.client.collections.get(collection_name)

#     def search(self, query: str, alpha: float = 0.5, limit: int = 3, top_k: int = 20, rerank: bool = False):
#         """Unified search with score and vector distance tracking."""
#         start_time = time.time()
#         fetch_count = max(top_k, limit) if rerank else limit

#         response = self.collection.query.hybrid(
#             query=query,
#             alpha=alpha,
#             limit=fetch_count,
#             rerank=wvc.Rerank(prop="content", query=query) if rerank else None,
#             # We now request both Score (Blended) and Distance (Vector Only)
#             return_metadata=wvc.MetadataQuery(score=True, distance=True)
#         )

#         results = response.objects[:limit]
#         self._display_summary(len(results), time.time() - start_time)
#         self._display_results(results)

#     def _display_summary(self, count: int, duration: float):
#         print(f"\nFound {count} matches in {duration:.3f} seconds.")
#         print("=" * 60)

#     def _display_results(self, objects):
#         if not objects:
#             print("No matches found.")
#             return
#         for i, obj in enumerate(objects):
#             self._print_single_object(i + 1, obj)

#     def _print_single_object(self, rank: int, obj):
#         props = obj.properties
#         # Score: Higher is better | Distance: Lower is better
#         score = obj.metadata.score or 0.0
#         dist = f"{obj.metadata.distance:.4f}" if obj.metadata.distance is not None else "N/A (Keyword match)"
        
#         print(f"Result #{rank}")
#         print(f" > Hybrid Score: {score:.4f} (Higher is better)")
#         print(f" > Vector Dist:  {dist} (Lower is better)")
#         print(f"File: {props.get('fileName')}")
#         print("-" * 30)
        
#         content = props.get('content', "")
#         preview = (content[:500] + '...') if len(content) > 500 else content
#         print(f"{preview}\n")

#     def close(self):
#         self.client.close()

#     def __enter__(self): return self
#     def __exit__(self, *args): self.close()

In [ ]:
#| notest
collection_name = 'math_writing'
if __name__ == "__main__":
    searcher = MathBrainSearcher(collection_name=collection_name)
    
    try:
        while True:
            user_query = input("\nAsk your Math Brain a question (or 'q' to quit): ")
            if user_query.lower() == 'q':
                break
            
            searcher.search(user_query, limit=10)
    finally:
        searcher.close()


Found 10 matches in 0.819 seconds.
Result #1
 > Hybrid Score: 0.5000 (Higher is better)
 > Vector Dist:  N/A (Keyword match) (Lower is better)
File: lemma_initial_or_final_object_in_a_category_that_is_also_in_a_full_subcategory_is_initial_or_final_in_the_subcategory.tex
------------------------------
\begin{lemma} \label{lemma:initial_or_final_object_in_a_category_that_is_also_in_a_full_subcategory_is_initial_or_final_in_the_subcategory}
    Let $\calC$ be a \CrefAndHyperrefIfExist{definition:full_subcategory_of_a_category}{full subcategory} of a \CrefAndHyperrefIfExist{definition:category}{(large) category} $\calD$. 
    Suppose that $\calD$ hsa an \CrefAndHyperrefIfExist{definition:initial_final_zero_objects_of_a_category}{initial object} $I$ (resp. a \CrefAndHyperrefIfExist{definition:ini...

Result #2
 > Hybrid Score: 0.4485 (Higher is better)
 > Vector Dist:  N/A (Keyword match) (Lower is better)
File: theorem_functor_from_category_of_topological_spaces_to_homotopy_category_of_to